In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import re
import os
from gspread_pandas import Spread, conf

In [2]:
config_path = r'C:\Users\USER\Downloads\Emarath_global\service_account.json'
emarath_global_sheet_url = "https://docs.google.com/spreadsheets/d/1IeqrQngvF8fZCwMyXzAR4J1dBkK02RRDtep2D1-yCe0/edit?"
c = conf.get_config(file_name=config_path)
spread_source = Spread(emarath_global_sheet_url, config=c)

df_sales = spread_source.sheet_to_df(sheet='Master_Lead_Data', index=0)
df_dispatch = spread_source.sheet_to_df(sheet='Master_Dispatch_Data', index=0)


In [3]:
# df_dispatch.to_excel("df_dispatch.xlsx")
df_dispatch.columns

Index(['COUNTRY', 'AGENT', 'DATE', 'TRACKING NUMBER', 'EM NUMBER', 'NAME',
       'NUMBER1', 'NUMBER2', 'STATE', 'ADDRESS', 'PRODUCT 1', 'QTY 1',
       'PRODUCT 2', 'QTY 2', 'VALUE', 'PAYMENT METHOD', '1', '2', '3',
       'DELIVERY AGENTS', 'STATUS', 'NOTES', 'REASON', 'CUSTOMER PATH', '',
       'DELIVERED/CANCELLED DATE_RTO', 'DEL/UNPAID DATE', 'DATESHIPMENT DATE',
       '3PL STATUS', '3PL REMARKS', 'FOLLOWUP AGENT NAME', 'VENDOR',
       'FOLLOWUP ASSIGNED DATE', '0.1'],
      dtype='str')

In [4]:
df_sales.head(2)

,EM NUMBER,COUNTRY,DATE,AGENT,CUSTOMER PATH,NAME,PHONE NO 1,PHONE NO 2,1,2,...,0.9,0.1,0.11,0.12,0.13,0.14,0.15,0.16,REF NO.2,COUNTRY.1
0,EMCR22501,KSA,04/02/2026,RAHIB,Missed lead,DECLINED,966565691657,,TRUE,FALSE,...,,,,,,,,,,
1,EMCR22502,KSA,04/02/2026,RAHIB,LEAD,SEKAR,919791383519,,TRUE,FALSE,...,,,,,,,,,,


In [5]:
# ── LOAD RAW FILES ──────────────────────────────────────────────
print("Loading files...")
# df_sales    = pd.read_csv('./Input/sales_data.csv')
# df_dispatch = pd.read_csv('./Input/dispatch_data.csv')
df_agents   = pd.read_csv('./Input/agent_details.csv')
df_product  = pd.read_csv('./Input/product_details.csv') 


Loading files...


In [ ]:
# ══════════════════════════════════════════════════════════════════
# NORMALISE COLUMN NAMES
# ══════════════════════════════════════════════════════════════════
def normalise_cols(df):
    df.columns = (
        df.columns
        .str.lower()
        .str.strip()
        .str.replace(r'\s+', '_', regex=True)
        .str.replace('/', '_', regex=False)
    )
    return df

df_sales    = normalise_cols(df_sales)
df_dispatch = normalise_cols(df_dispatch)

# ══════════════════════════════════════════════════════════════════
# DROP JUNK COLUMNS
# Removes blank-name cols and purely numeric cols (0.1, 919961190801 etc.)
# ══════════════════════════════════════════════════════════════════
def drop_junk_columns(df, label='df'):
    junk = [
        c for c in df.columns
        if c.strip() == ''
        or re.fullmatch(r'[\d.]+', c)
    ]
    if junk:
        print(f"  [{label}] Dropping junk columns: {junk}")
        df = df.drop(columns=junk)
    return df

df_sales    = drop_junk_columns(df_sales,    'sales')
df_dispatch = drop_junk_columns(df_dispatch, 'dispatch')

print(f"\nSales columns    : {list(df_sales.columns)}")
print(f"Dispatch columns : {list(df_dispatch.columns)}")

# ══════════════════════════════════════════════════════════════════
# RENAME SALES COLUMNS TO STANDARD NAMES
# Actual observed names: phone_no_1, phone_no_2, dispatch_date, agent_name
# ══════════════════════════════════════════════════════════════════
df_sales.rename(columns={
    'phone_no_1':    'phone_number',       
    'phone_no_2':    'phone_number_2',     
    'dispatch_date': 'sale_dispatch_date', 
    'agent_name':    'sale_agent_name',    
}, inplace=True)

# ══════════════════════════════════════════════════════════════════
# RENAME DISPATCH COLUMNS
# Actual observed names: status → delivery_status
# delivered_cancel_rto_date and del_unpaid_date already correct after normalise
# ══════════════════════════════════════════════════════════════════
df_dispatch.rename(columns={
    'status': 'delivery_status', 

}, inplace=True)

print(f"Dispatch columns after rename: {list(df_dispatch.columns)}")


# ══════════════════════════════════════════════════════════════════
required_sales = ['em_number', 'date', 'agent', 'status']

missing_req = [c for c in required_sales if c not in df_sales.columns]
if missing_req:
    raise ValueError(f"Sales file missing required columns: {missing_req}")

mask_dirty = df_sales[required_sales].isnull().any(axis=1)
df_errors  = df_sales[mask_dirty].copy()
df_sales   = df_sales[~mask_dirty].copy()

os.makedirs('./Output', exist_ok=True)
df_errors.to_csv('./Output/error_log.csv', index=False)
print(f"\nQuarantined {len(df_errors)} dirty rows → error_log.csv")
print(f"Clean sales rows: {len(df_sales)}")

# ══════════════════════════════════════════════════════════════════
# PARSE DATES
# ══════════════════════════════════════════════════════════════════
df_sales['date'] = pd.to_datetime(df_sales['date'], dayfirst=True, errors='coerce')
# df_sales['sale_dispatch_date'] = pd.to_datetime(df_sales['sale_dispatch_date'], dayfirst=True, errors='coerce')

df_dispatch['dispatch_date'] = pd.to_datetime(df_dispatch['date'], dayfirst=True, errors='coerce')
df_dispatch['delivered_cancelled_rto_date'] = pd.to_datetime(df_dispatch['delivered_cancelled_date_rto'], dayfirst=True, errors='coerce')
df_dispatch['del_unpaid_date'] = pd.to_datetime(df_dispatch['del_unpaid_date'], dayfirst=True, errors='coerce')
df_dispatch['dateshipment_date'] = pd.to_datetime(df_dispatch['dateshipment_date'], dayfirst=True, errors='coerce')
df_dispatch['followup_assigned_date'] = pd.to_datetime(df_dispatch['followup_assigned_date'], dayfirst=True, errors='coerce')

# ══════════════════════════════════════════════════════════════════
# PHONE CLEANING
# ══════════════════════════════════════════════════════════════════
def clean_phone(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.nan
    s = str(val).strip()
    if s == '' or s.lower() == 'nan':
        return np.nan

    s_stripped = re.split(
        r'\s+(?:ext|extension|x|extn)[\s.\-#]*\d',
        s, flags=re.IGNORECASE
    )[0].strip()

    separators_removed = re.sub(r'[\s\-.()+]', '', s_stripped)
    if separators_removed.isdigit():
        digits = separators_removed
    else:
        digit_groups = re.findall(r'\d+', s_stripped)
        if not digit_groups:
            return np.nan
        lone_match = [g for g in digit_groups if 7 <= len(g) <= 13]
        digits = lone_match[0] if lone_match else ''.join(digit_groups)

    return digits if 7 <= len(digits) <= 13 else np.nan


df_sales['phone_number_raw']   = df_sales['phone_number'].copy()
df_sales['phone_number_2_raw'] = (
    df_sales['phone_number_2'].copy()
    if 'phone_number_2' in df_sales.columns else np.nan
)

df_sales['phone_number'] = df_sales['phone_number'].apply(clean_phone)
if 'phone_number_2' in df_sales.columns:
    df_sales['phone_number_2'] = df_sales['phone_number_2'].apply(clean_phone)

df_sales['phone_invalid_flag'] = (
    df_sales['phone_number'].isna() &
    df_sales['phone_number_raw'].notna() &
    df_sales['phone_number_raw'].astype(str).str.strip().ne('') &
    df_sales['phone_number_raw'].astype(str).str.lower().ne('nan')
).astype(int)

n_valid   = df_sales['phone_number'].notna().sum()
n_invalid = df_sales['phone_invalid_flag'].sum()
n_blank   = df_sales['phone_number_raw'].isna().sum()

print(f"\nPhone cleaning results:")
print(f"  Valid numbers extracted   : {n_valid:,}")
print(f"  Unextractable (no digits) : {n_invalid:,}")
print(f"  Were blank originally     : {n_blank:,}")

mixed_mask = (
    df_sales['phone_number_raw'].astype(str).str.contains(r'[a-zA-Z]', na=False) &
    df_sales['phone_number'].notna()
)
if mixed_mask.sum() > 0:
    print(f"\n  Sample mixed values where digits were extracted:")
    for _, row in df_sales.loc[mixed_mask, ['phone_number_raw', 'phone_number']].head(10).iterrows():
        print(f"    '{row['phone_number_raw']}'  →  '{row['phone_number']}'")

invalid_mask = df_sales['phone_invalid_flag'] == 1
if invalid_mask.sum() > 0:
    print(f"\n  Sample values cleared (no valid digits):")
    for v in df_sales.loc[invalid_mask, 'phone_number_raw'].head(10).values:
        print(f"    '{v}' → NaN")

# ══════════════════════════════════════════════════════════════════
# STANDARDISE STATUS
# ══════════════════════════════════════════════════════════════════
df_sales['status_raw'] = df_sales['status'].copy()
df_sales['status'] = (
    df_sales['status']
    .astype(str).str.lower().str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

STATUS_MAP = {
    'won':               'won',
    'super hot':         'super hot',
    'hot':               'hot',
    'warm':              'warm',
    'whats app engage':  'whatsapp engaged',
    'whatsapp engage':   'whatsapp engaged',
    'whatsapp engaged':  'whatsapp engaged',
    'cold':              'cold',
    'missed lead':       'cold',
    'lead':              'new enquiry',
    'new enquiry':       'new enquiry',
}
df_sales['status'] = df_sales['status'].map(STATUS_MAP).fillna(df_sales['status'])

FUNNEL_RANK = {
    'new enquiry': 1, 'whatsapp engaged': 2, 'warm': 3,
    'hot': 4, 'super hot': 5, 'won': 6, 'cold': 0,
}
df_sales['funnel_rank'] = df_sales['status'].map(FUNNEL_RANK).fillna(0).astype(int)

# ── CONFLICT FLAG ─────────────────────────────────────────────────
if 'customer_path' in df_sales.columns:
    conflict_mask = (
        (df_sales['status'] == 'won') &
        (df_sales['customer_path'].astype(str).str.upper().str.strip() == 'MISSED LEAD')
    )
else:
    conflict_mask = pd.Series(False, index=df_sales.index)

df_sales['data_conflict_flag'] = conflict_mask.astype(int)
n_conflicts = int(conflict_mask.sum())
if n_conflicts:
    print(f"\nWARNING: {n_conflicts} rows status=Won but source=Missed Lead → flagged.")

# ── STANDARDISE SOURCE ────────────────────────────────────────────
SOURCE_MAP = {
    'LEAD':        'Lead',
    'MISSED LEAD': 'Missed Lead',
    'NEW ENQUIRY': 'New Enquiry',
}
if 'customer_path' in df_sales.columns:
    df_sales['source'] = (
        df_sales['customer_path']
        .astype(str).str.upper().str.strip()
        .map(SOURCE_MAP)
        .fillna(df_sales['customer_path'])
    )
else:
    df_sales['source'] = 'Unknown'

# ── STANDARDISE COUNTRY ───────────────────────────────────────────
COUNTRY_MAP = {
    'KSA': 'Saudi Arabia', 'SAUDI': 'Saudi Arabia', 'SAUDI ARABIA': 'Saudi Arabia',
    'UAE': 'UAE', 'OMAN': 'Oman', 'KUWAIT': 'Kuwait',
    'BAHRAIN': 'Bahrain', 'QATAR': 'Qatar',
}
df_sales['country'] = (
    df_sales['country']
    .fillna('UAE')
    .astype(str).str.strip().str.upper()
    .map(COUNTRY_MAP)
    .fillna('UAE')
)

# ── CURRENCY → AED ────────────────────────────────────────────────
RATES = {
    'UAE': 1.000, 'Saudi Arabia': 0.978, 'Oman': 9.530,
    'Kuwait': 11.950, 'Bahrain': 9.750, 'Qatar': 1.010,
}
df_sales['value']     = pd.to_numeric(df_sales['value'], errors='coerce')
df_sales['value_aed'] = df_sales.apply(
    lambda r: r['value'] * RATES.get(r['country'], 1.0)
    if pd.notna(r['value']) else np.nan,
    axis=1
)

avg_won_aed = df_sales.loc[df_sales['status'] == 'won', 'value_aed'].mean()
avg_won_aed = avg_won_aed if pd.notna(avg_won_aed) else 0.0
print(f"\nAverage Won deal value: AED {avg_won_aed:.2f}")

df_sales['is_estimated'] = df_sales['value_aed'].isna().astype(int)
df_sales['value_aed'] = df_sales['value_aed'].fillna(avg_won_aed)

# ── PROBABILITY & WEIGHTED FORECAST ──────────────────────────────
PROB_MAP = {
    'won': 1.00, 'booking': 0.90, 'super hot': 0.80, 'hot': 0.50,
    'warm': 0.20, 'whatsapp engaged': 0.10,
    'new enquiry': 0.05, 'cold': 0.00,
}
df_sales['probability']       = df_sales['status'].map(PROB_MAP).fillna(0.05)
df_sales['weighted_forecast'] = df_sales['value_aed'] * df_sales['probability']
df_sales['is_leakage']        = df_sales['status'].isin(['cold']).astype(int)

# ══════════════════════════════════════════════════════════════════
# PROCESS DISPATCH
# ══════════════════════════════════════════════════════════════════
df_dispatch['delivery_status'] = (
    df_dispatch['delivery_status'].astype(str).str.strip().str.upper()
)
df_dispatch['rto_flag']      = df_dispatch['delivery_status'].str.contains('RTO', na=False).astype(int)
df_dispatch['is_delivered']  = df_dispatch['delivery_status'].str.contains('DELIVERED', na=False).astype(int)
df_dispatch['is_cod_unpaid'] = df_dispatch['delivery_status'].str.contains('UNPAID', na=False).astype(int)
df_dispatch['is_sales_closed'] = df_dispatch['delivery_status'].str.contains('SALE CLOSED', na=False).astype(int)

# days_in_transit: dispatch date → DEL/UNPAID DATE (actual delivery)
df_dispatch['days_in_transit'] = (
    df_dispatch['delivered_cancelled_rto_date'] - df_dispatch['dispatch_date']
).dt.days

# bonus: days from delivery → RTO/cancellation
df_dispatch['days_delivered_to_closed'] = (
    df_dispatch['del_unpaid_date'] - df_dispatch['delivered_cancelled_rto_date']
).dt.days

# ══════════════════════════════════════════════════════════════════
# MERGE on em_number
# ══════════════════════════════════════════════════════════════════
df_master = df_sales.merge(df_dispatch, on='em_number', how='left')
print("df_master columns \n", df_master.columns)

assert 'date_x' in df_master.columns, \
    "FATAL: 'date' (order date) missing after merge."

# days from order date → actual dispatch
df_master['days_to_ship'] = (df_master['dispatch_date'] - df_master['date_x']).dt.days

print(f"\nMaster table rows          : {len(df_master):,}")
print(f"Rows with dispatch data    : {df_master['dispatch_date'].notna().sum():,}")
print(f"RTO (returned) orders      : {int(df_master['rto_flag'].sum()):,}")
print(f"COD unpaid orders          : {int(df_master['is_cod_unpaid'].sum()):,}")

# ══════════════════════════════════════════════════════════════════
# DIM_DATE  (full calendar 2026)
# ══════════════════════════════════════════════════════════════════
all_dates = pd.date_range('2026-01-01', '2026-12-31', freq='D')
dim_date  = pd.DataFrame({'date': all_dates})
dim_date['day']        = dim_date['date'].dt.day
dim_date['month_num']  = dim_date['date'].dt.month
dim_date['month_name'] = dim_date['date'].dt.strftime('%B')
dim_date['quarter']    = dim_date['date'].dt.quarter
dim_date['year']       = dim_date['date'].dt.year
dim_date['week_no']    = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['year_month'] = dim_date['date'].dt.to_period('M').astype(str)
dim_date['is_weekend'] = dim_date['date'].dt.dayofweek.isin([4, 5]).astype(int)  # Fri+Sat

PEAK_DATES = [
    '2026-02-13', '2026-02-14', '2026-02-15',
    '2026-03-20', '2026-03-21', '2026-03-22', '2026-03-23',
    '2026-05-27', '2026-05-28', '2026-05-29', '2026-05-30',
    '2026-07-17', '2026-07-18',
    '2026-09-23', '2026-09-26', '2026-09-27',
    '2026-12-01', '2026-12-02', '2026-12-03',
    '2026-12-20', '2026-12-21', '2026-12-22',
    '2026-12-23', '2026-12-24', '2026-12-25',
    '2026-12-31',
]
dim_date['is_peak_season'] = dim_date['date'].dt.strftime('%Y-%m-%d').isin(PEAK_DATES).astype(int)
dim_date['is_peak_month']  = dim_date['month_num'].isin([2, 3, 5, 9, 12]).astype(int)

# ══════════════════════════════════════════════════════════════════
# DIM_AGENT
# ══════════════════════════════════════════════════════════════════
dim_agent = (
    df_sales[['agent']].drop_duplicates(subset=['agent'])
    .reset_index(drop=True)
    .rename(columns={'agent': 'agent_name'})
)
dim_agent['agent_id']          = range(1, len(dim_agent) + 1)
dim_agent['employment_status'] = 'Active'
dim_agent['resign_date']       = pd.NaT

RESIGNED_AGENTS = {
    'SHYAMJIL':   '2026-02-04',
    'ADHIL':      '2026-02-14',
    'BURHANA':    '2026-03-12',
    'ARUN':       '2026-03-07',
    'NAJA':       '2026-02-12',
    'FARSANA':    '2026-02-19',
    'SHABNA':     '2026-02-19',
    'SUHANTHIKA': '2026-03-03',
    'AJIN':       '2026-02-24',
    'GOWTHAM':    '2026-03-07',
    'RINSIYA':    '2026-03-19',
    'NAFI':       '2026-03-04',
    'ARSHAD':     '2026-04-04',
    'SHAMNA':     '2026-04-04',
    'RAHIB':      '2026-04-04',
}
for agent_name, resign_date in RESIGNED_AGENTS.items():
    mask = dim_agent['agent_name'] == agent_name
    dim_agent.loc[mask, 'employment_status'] = 'Resigned'
    dim_agent.loc[mask, 'resign_date']       = (
        pd.to_datetime(resign_date) if resign_date else pd.NaT
    )

try:
    df_agents = normalise_cols(df_agents)
    agent_key = 'agent_name' if 'agent_name' in df_agents.columns else df_agents.columns[0]
    dim_agent = dim_agent.merge(df_agents, left_on='agent_name', right_on=agent_key, how='left')
    print(f"\nAgent details merged. Columns: {list(dim_agent.columns)}")
except Exception as e:
    dim_agent['monthly_salary']  = 0
    dim_agent['commission_rate'] = 0
    print(f"\nNote: agent_details.csv not merged ({e}). Fill salary manually.")

# ══════════════════════════════════════════════════════════════════
# DIM_PRODUCT
# FIX: sales uses product_1 + product_2 columns, not a single 'product' column
# ══════════════════════════════════════════════════════════════════
products_sales = []
for col in ['product_1', 'product_2']:
    if col in df_sales.columns:
        products_sales += df_sales[col].dropna().unique().tolist()

products_dispatch = []
for col in ['product_1', 'product_2']:
    if col in df_dispatch.columns:
        products_dispatch += df_dispatch[col].dropna().unique().tolist()

all_prods   = list(set(products_sales + products_dispatch))
dim_product = pd.DataFrame({'product': all_prods})
dim_product['product_id'] = range(1, len(dim_product) + 1)

try:
    df_product = normalise_cols(df_product)
    prod_key   = 'product' if 'product' in df_product.columns else df_product.columns[0]
    dim_product = dim_product.merge(df_product, on=prod_key, how='left')
    print(f"Product details merged. Columns: {list(dim_product.columns)}")
except Exception as e:
    dim_product['category']   = 'Other'
    dim_product['cost_price'] = 0
    print(f"Note: product_details.csv not merged ({e}). Fill category manually.")

# ══════════════════════════════════════════════════════════════════
# DIM_COURIER
# FIX: actual column is 'delivery_agents' not 'delivery_agent' — detect dynamically
# ══════════════════════════════════════════════════════════════════
courier_col = next(
    (c for c in df_dispatch.columns if 'delivery_agent' in c),
    None
)
if courier_col:
    dim_courier = (
        df_dispatch[[courier_col]].dropna()
        .drop_duplicates().reset_index(drop=True)
    )
    dim_courier.columns       = ['courier_name']
    dim_courier['courier_id'] = range(1, len(dim_courier) + 1)
    dim_courier.to_csv('./Output/dim_courier.csv', index=False)
    print(f"\ndim_courier.csv → {len(dim_courier)} couriers")
else:
    print("\nNote: No delivery_agent column found in dispatch — dim_courier skipped.")

# ══════════════════════════════════════════════════════════════════
# EXPORT
# ══════════════════════════════════════════════════════════════════
df_master.to_csv('./Output/fact_sales.csv',      index=False)
df_dispatch.to_csv('./Output/fact_dispatch.csv', index=False)
dim_date.to_csv('./Output/dim_date.csv',         index=False)
dim_agent.to_csv('./Output/dim_agent.csv',       index=False)
dim_product.to_csv('./Output/dim_product.csv',   index=False)

# ══════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════
total_won      = df_sales.loc[df_sales['status'] == 'won', 'value_aed'].sum()
total_pipeline = df_sales.loc[
    df_sales['status'].isin(['super hot', 'hot', 'warm', 'whatsapp engaged']),
    'weighted_forecast'
].sum()
total_leaked = df_sales.loc[df_sales['status'] == 'cold', 'value_aed'].sum()

avg_transit = df_dispatch['days_in_transit'].mean()
avg_to_rto  = df_dispatch['days_delivered_to_rto'].mean()
avg_to_ship = df_master['days_to_ship'].mean()

print("\n" + "=" * 60)
print("OUTPUT FILES → ./Output/")
print("=" * 60)
print(f"  fact_sales.csv      {len(df_master):>8,} rows")
print(f"  fact_dispatch.csv   {len(df_dispatch):>8,} rows")
print(f"  dim_date.csv        {len(dim_date):>8,} rows (2026 only)")
print(f"  dim_agent.csv       {len(dim_agent):>8,} agents")
print(f"  dim_product.csv     {len(dim_product):>8,} products")
print(f"  error_log.csv       {len(df_errors):>8,} quarantined rows")
print()
print("STATUS BREAKDOWN:")
print(df_sales['status'].value_counts().to_string())
print()
print("FINANCIAL SUMMARY (AED):")
print(f"  Actual won revenue  : {total_won:>14,.2f}")
print(f"  Weighted pipeline   : {total_pipeline:>14,.2f}")
print(f"  Estimated leakage   : {total_leaked:>14,.2f}")
print(f"  Conflict rows       : {n_conflicts:>14,}")
print()
print("DISPATCH METRICS:")
print(f"  Avg days dispatch → delivery  : {avg_transit:.1f} days")
print(f"  Avg days delivery → RTO       : {avg_to_rto:.1f} days")
print(f"  Avg days order → dispatch     : {avg_to_ship:.1f} days")
print()
print("NEXT STEPS:")
print("  1. Open ./Output/dim_agent.csv   → fill monthly_salary + commission_rate")
print("  2. Open ./Output/dim_agent.csv   → verify RESIGNED_AGENTS names + dates")
print("  3. Open ./Output/dim_product.csv → fill category + cost_price")
print("  4. Load all Output CSVs into Power BI")
print("  5. In Model View, create relationships:")
print("       fact_sales[date]            → dim_date[date]")
print("       fact_sales[agent]           → dim_agent[agent_name]")
print("       fact_sales[product_1]       → dim_product[product]")
print("       fact_sales[em_number]       → fact_dispatch[em_number]")


Sales columns    : ['em_number', 'country', 'date', 'agent', 'customer_path', 'name', 'phone_number', 'phone_number_2', 'product_1', 'qty', 'call_status', 'notes', 'status', 'gender', 'language', 'city', 'national_code_ksa', 'address', 'product_2', 'qty.1', 'value', 'payment_method', 'lead_source', 'dispatch', 'sale_dispatch_date', 'sale_agent_name', 'cs_remarks', 'campaign_id', 'source_sheet', 'vendor', 'vendor-product', 'ref_no.1', 'ref_no.2', 'country.1', 'phone_number_raw', 'phone_number_2_raw', 'phone_invalid_flag', 'status_raw', 'funnel_rank', 'data_conflict_flag', 'source', 'value_aed', 'is_estimated', 'probability', 'weighted_forecast', 'is_leakage']
Dispatch columns : ['country', 'agent', 'date', 'tracking_number', 'em_number', 'name', 'number1', 'number2', 'state', 'address', 'product_1', 'qty_1', 'product_2', 'qty_2', 'value', 'payment_method', 'delivery_agents', 'delivery_status', 'notes', 'reason', 'customer_path', 'delivered_cancelled_date_rto', 'del_unpaid_date', 'dates

C:\Users\USER\AppData\Local\Temp\ipykernel_1808\4040728950.py:276: PerformanceWarning: Adding/subtracting object-dtype array to DatetimeArray not vectorized.
  df_dispatch['delivered_cancelled_rto_date'] - df_dispatch['date']


TypeError: unsupported operand type(s) for -: 'Timestamp' and 'str'